# 01 — Batched Mai Chi data on T4 → Google Drive

Run this notebook FIRST on T4. It creates data, not student training. Then open [02 — RTX 6000 training](https://colab.research.google.com/github/Huyvux12/Ancaptritue/blob/main/notebooks/02_MaiChi_Train_RTX6000.ipynb).

The public ZeroTTS export allows batched text/global-transformer/codec execution, but its local sampler still processes one utterance per call. This is partial model batching, not fully vectorized sampling or a demonstrated T4 speedup. Start at batch 2; memory errors halve the batch without switching devices or sampling parameters.

Local SSD holds active WAVs/features. Checksummed immutable tar shards are saved to mounted Drive after each 128-text block. A COMPLETE bundle is published only after features are ready. Runtime loss can lose the current unsynced block.

## 1. Configure data and storage

Use the same mode/Drive folder in notebook 02. Full requests 20,000 texts; pilot checks execution only. Reserve approximately 60 GB or more on local/Drive storage depending on utterance lengths and caches. Check Drive quota separately.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
os.environ["ORT_DISABLE_TELEMETRY"] = "1"  # Before ONNX Runtime initialization
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
MODE = "pilot"  # SAME mode in both notebooks; "full" for the main experiment
USE_DRIVE = True
DRIVE_PARENT = "/content/drive/MyDrive/MaiChi_Distillation_v03"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
# External runtime: USE_DRIVE=False, DRIVE_PARENT=an already mounted/synced Drive folder
DATA_BUNDLE_DIR = Path(DRIVE_PARENT) / "datasets" / MODE
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORK)
assert MODE in {"pilot", "full"}
ROOT = WORK / "maichi_data_v03" / MODE
ROOT.mkdir(parents=True, exist_ok=True)
TEXT_COUNT = 512 if MODE == "pilot" else 20000
TEACHER_DEVICE = "cuda"
TEACHER_BATCH_SIZE = 2  # T4: try 4 only after measuring memory/throughput
CPU_THREADS = 2
FEATURE_WORKERS = 2
SYNC_EVERY = 128
RUN_ASR_AUDIT = True
RUN_REAL_BATCH_CHECK = True
RUN_TEACHER_BENCHMARK = MODE == "full"
print({"local_root": str(ROOT), "drive_bundle": str(DATA_BUNDLE_DIR), "texts": TEXT_COUNT})
print("Local free GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 2. Clone pinned source

Resume automatically reuses the data producer commit. Existing source edits are protected; use a new dataset folder to change the implementation/sampling/text selection.

In [ ]:
REPO_URL = "https://github.com/Huyvux12/Ancaptritue.git"
p = DATA_BUNDLE_DIR / "bundle.json"
saved_manifest = json.loads(p.read_text()) if p.exists() else {}
REPO_REF = saved_manifest.get("source_commit") or "main"
REPO_DIR = WORK / "Ancaptritue"
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Different repository"
    assert not subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip(), "Preserve source edits before updating"
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
import hashlib
SOURCE_FILES = ["maichi_pipeline.py", "maichi_flow.py", "maichi_teacher_batch.py", "maichi_handoff.py", "maichi_train_runner.py"]
SOURCE_HASHES = {name: hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() for name in SOURCE_FILES}
REVISION = {"repository": REPO_URL, "commit": REPO_SHA, "files": SOURCE_HASHES}
print("Pinned source:", REPO_SHA)


## 3. Install and check CUDA

Keep runtime CUDA PyTorch. Teacher uses its original FP32 graphs. Telemetry is disabled before ONNX initialization; no FP16 conversion or sampling change is performed.

ONNX GPU is pinned to 1.23.2 for CUDA 12 PyTorch or 1.30.0 for CUDA 13 PyTorch; PyTorch itself is retained. Actual GPU execution is verified below.

In [ ]:
def pip_install(*packages, extra=None):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(packages)
    if extra:
        cmd += list(extra)
    subprocess.run(cmd, check=True)

pip_install("setuptools==80.9.0", "wheel", "cython==3.0.12", "numpy==2.2.6")
pip_install(extra=["-r", str(REPO_DIR / "requirements-colab.txt")])
os.environ["CC"], os.environ["CXX"] = "gcc", "g++"
pip_install("pyworld==0.3.5", extra=["--no-build-isolation"])
pip_install("zerotts==0.1.5", extra=["--no-deps"])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=True)
if TEACHER_DEVICE == "cuda":
    import torch
    cuda_major = int(torch.version.cuda.split(".")[0]) if torch.version.cuda else 0
    assert cuda_major in {12, 13}, "Use a CUDA 12/13 PyTorch runtime for teacher GPU inference"
    ORT_GPU_PIN = "1.30.0" if cuda_major == 13 else "1.23.2"
    pip_install(f"onnxruntime-gpu=={ORT_GPU_PIN}")
    print("ONNX GPU pin for CUDA", cuda_major, ":", ORT_GPU_PIN)
else:
    pip_install("onnxruntime==1.23.2")


In [ ]:
import torch, numpy as np, onnxruntime as ort
ort.disable_telemetry_events()
assert torch.cuda.is_available(), "No GPU visible to this kernel. Connect the intended GPU runtime first."
# Real CUDA operations catch missing kernel/architecture support early.
x = torch.randn(512, 512, device="cuda")
_ = x @ x
_ = torch.nn.Conv1d(16, 16, 3).cuda()(torch.randn(2, 16, 128, device="cuda"))
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Torch/CUDA:", torch.__version__, torch.version.cuda)
print("ONNX providers:", ort.get_available_providers())
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()  # Reuse compatible CUDA/cuDNN libraries loaded by PyTorch.
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
(ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


## 4. Restore partial Drive data

Archives are verified and paths rebound to this runtime. Partial bundles are allowed only in notebook 01. Local WAV sidecars also allow resume after a normal cell interruption.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
for name in ["maichi_pipeline", "maichi_flow", "maichi_teacher_batch", "maichi_handoff", "maichi_train_runner"]:
    sys.modules.pop(name, None)
import maichi_pipeline as mp
import maichi_handoff as handoff

def launch(filename, stage=None, **kwargs):
    cmd = [sys.executable, "-u", str(REPO_DIR / filename)]
    if stage is not None: cmd.append(stage)
    cmd += ["--root", str(ROOT)]
    for key, value in kwargs.items():
        flag = "--" + key.replace("_", "-")
        if isinstance(value, bool):
            if value: cmd.append(flag)
        elif value is not None: cmd += [flag, str(value)]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

def data_run(stage, **kwargs): return launch("maichi_pipeline.py", stage, **kwargs)
if (DATA_BUNDLE_DIR / "bundle.json").exists(): handoff.restore(DATA_BUNDLE_DIR, ROOT, require_complete=False)
p = ROOT / "source_revision.json"
if p.exists(): assert json.loads(p.read_text())["files"] == SOURCE_HASHES, "Source changed; use a new dataset folder"
mp.atomic_json(p, REVISION)
for filename in SOURCE_FILES + ["LICENSE_VITS.txt", "LICENSE_ZEROTTS.txt"]: shutil.copy2(REPO_DIR / filename, ROOT / filename)


## 5. Teacher and actual batch smoke check

Listen before generating the corpus. The real check compares batch-one against the upstream sampler, then synthesizes independent texts of different lengths in batch. Passing verifies execution, not pronunciation quality.

In [ ]:
if not (ROOT / "smoke.json").exists(): data_run("smoke", device=TEACHER_DEVICE, threads=CPU_THREADS)
if RUN_REAL_BATCH_CHECK:
    subprocess.run([sys.executable, "-u", str(REPO_DIR / "test_maichi_teacher_batch.py"), "--real", "--device", TEACHER_DEVICE,
                    "--output-dir", str(ROOT / "batch_smoke")], check=True)
from IPython.display import Audio, display
for row in mp.read_jsonl(ROOT / "batch_smoke/results.jsonl"):
    print(row["text"]); display(Audio(filename=str(ROOT / "batch_smoke" / row["file"])))


## 6. Reserve benchmark, select text

Reserve all 137 test items before selecting training transcripts. Existing selection is reused. Exact/substring filtering does not exclude all paraphrases/shared sources.

In [ ]:
if not (ROOT / "benchmark_manifest.jsonl").exists(): data_run("benchmark")
assert len(mp.read_jsonl(ROOT / "benchmark_manifest.jsonl")) == 137
if not (ROOT / "texts.jsonl").exists(): data_run("texts", count=TEXT_COUNT)
assert len(mp.read_jsonl(ROOT / "texts.jsonl")) == TEXT_COUNT, "Mode/count mismatch"
handoff.publish(ROOT, DATA_BUNDLE_DIR, complete=False)


## 7. Batched teacher corpus generation

Length bucketing is within each checkpoint block. Each text has an independent RNG; numerical differences across hardware/batch shapes may still change sampled output. Logs record actual batch throughput. The local sampler is per lane; global transformer and codec are batched. Rejected/finished items are recorded and reused on resume.

In [ ]:
launch("maichi_teacher_batch.py", batch_size=TEACHER_BATCH_SIZE, device=TEACHER_DEVICE, threads=CPU_THREADS,
       sync_dir=DATA_BUNDLE_DIR, sync_every=SYNC_EVERY)
print(json.loads((ROOT / "corpus_summary.json").read_text()))


## 8. Audit and features

WORLD F0 extraction runs on CPU. Partial local features are reusable, but VM loss before final publication may require re-extracting them. Drive-saved teacher WAVs are reused. Vocabulary is built from TRAIN text only.

In [ ]:
if RUN_ASR_AUDIT:
    data_run("audit", count=32 if MODE == "full" else 8, device="cuda")
    import pandas as pd
    display(pd.read_json(ROOT / "corpus_asr_audit.jsonl", lines=True).sort_values("wer", ascending=False).head(8))
data_run("features", workers=FEATURE_WORKERS)


## 9. Optional teacher benchmark on T4

Save 137 teacher WAVs so notebook 02 needs no teacher model. This baseline uses single-request inference. T4 teacher RTF must not be reported as a same-hardware speed comparison against the RTX student.

In [ ]:
if RUN_TEACHER_BENCHMARK:
    data_run("benchmark")  # Restore the pinned evaluator checkout after VM loss
    data_run("synthesize", model="teacher", device=TEACHER_DEVICE, threads=CPU_THREADS)
    mp.atomic_json(ROOT / "benchmark_wavs/teacher_maichi/hardware.json", {
        "gpu": torch.cuda.get_device_name(0), "device": TEACHER_DEVICE, "timing_protocol": "single request on producer T4"})


## 10. Publish COMPLETE bundle

Wait for READY FOR NOTEBOOK 02. Shards contain WAVs, features, manifests, vocabulary, provenance and optional benchmark references. Teacher weights are not uploaded. Immutable data objects/partial metadata snapshots are retained for resume; keep enough Drive space.

In [ ]:
bundle = handoff.publish(ROOT, DATA_BUNDLE_DIR, complete=True)
print(json.dumps({"status": "READY FOR NOTEBOOK 02", "accepted": bundle["accepted"], "bundle_id": bundle["bundle_id"],
                  "source_commit": bundle["source_commit"], "drive_folder": str(DATA_BUNDLE_DIR)}, indent=2))


## Next

Open notebook 02 on RTX 6000 with the same mode/Drive folder. It restores data to SSD and trains without regenerating the corpus.

Real teacher batching is CPU-tested before publication. T4 memory/performance remain unmeasured; this notebook runs a CUDA batch smoke check on your T4. The public local sampler is partly serialized, so larger batches are not guaranteed to be faster.